In [0]:
from pyspark.sql.functions import to_date, current_date, col

# Step-1: Read from Bronze (Streaming)
# 
cdc_df = spark.readStream.table("retails.silver.categories_cdc")


In [0]:
from pyspark.sql.functions import to_json, struct, when, col

# Step 4: Apply Type Casting & Standardization
cdc_df = cdc_df.withColumn("category_id", col("category_id").cast("integer")) \
                    .withColumn("category_department_id", col("category_department_id").cast("integer")) \
                    .withColumn("category_name", col("category_name").cast(("string"))) \
                    .withColumn("record_hash", col("record_hash"))
                        


In [0]:
# make ready to upsert silver table categories_cleaned
silver_df = cdc_df \
            .withColumn("category_id", col("category_id").cast("bigint")) \
            .withColumn("category_department_id", col("category_department_id").cast("bigint")) \
            .withColumn("batch_id", col("batch_id").cast("string"))


In [0]:
# upsertcategories_cleaned table
MERGE_UPDATE = """
    MERGE INTO retails.silver.categories_cleaned t
    USING global_temp.categories_cleaned_vw s
    ON t.category_id = s.category_id
    WHEN MATCHED AND s.record_hash != t.record_hash AND s.op='UPDATE' THEN
        UPDATE SET t.category_name = s.category_name, t.category_department_id = s.category_department_id, t.is_deleted = false, t.updated_ts = current_timestamp(), t.op= s.op, t.record_hash = s.record_hash
    
    WHEN MATCHED AND s.is_deleted AND s.op='DELETE' THEN
        UPDATE SET t.is_deleted = true, t.updated_ts = current_timestamp(), t.op= s.op
    
    WHEN NOT MATCHED AND s.op='INSERT' THEN
        INSERT (
            category_id,
            category_name,
            category_department_id,
            is_deleted,
            ingestion_ts,
            ingestion_dt,
            source_system,
            source_file_name,
            batch_id,
            run_id,
            op,
            record_hash,
            created_ts,
            updated_ts
                    )
        VALUES(
            s.category_id,
            s.category_name,
            s.category_department_id,
            s.is_deleted,
            s.ingestion_ts,
            s.ingestion_dt,
            s.source_system,
            s.source_file_name,
            s.batch_id,
            s.run_id,
            s.op,
            s.record_hash,
            current_timestamp(),
            current_timestamp()
            )
    """

# spark.sql(merge_query).show()



In [0]:
def upsert_to_silver(batch_df, batch_id):

    batch_df = batch_df.cache()
    batch_df.createOrReplaceGlobalTempView(
        "categories_cleaned_vw"
    )
    spark.sql(MERGE_UPDATE)

    batch_df.unpersist()

In [0]:
silver_df = silver_df.drop("event_ts")
# silver_df.printSchema()

In [0]:

silver_df.writeStream \
    .format("delta") \
    .foreachBatch(upsert_to_silver) \
    .option("checkpointLocation", "dbfs:/Volumes/data/raw/_checkpoints/retails/dev/silver/categories_cleaned/") \
    .trigger(once=True) \
    .start() \
    .awaitTermination()

In [0]:
# quarantine_status	Meaning
# NEW	Newly quarantined record, not yet reviewed
# INVALID	Permanently invalid record
# RESCUED	Record has rescued_data but may be recoverable
# CORRUPT	Completely corrupt JSON/file structure
# PARTIAL_VALID	Some columns valid, some problematic
# FIXED	Data corrected successfully
# REPROCESSED	Re-ingested into clean table

In [0]:
# GRANTS = "ALTER TABLE retails.silver.categories_quarantine OWNER TO `dp-sales-engineers`"
# spark.sql(GRANTS).display()